In [1]:
from datetime import datetime, timedelta
from enum import Enum

class PaymentStatus(Enum):
    PENDING = "pending"
    COMPLETED = "completed"
    OVERDUE = "overdue"
    FAILED = "failed"

class Payment:
    _transaction_id_counter = 9000
    PENALTY_RATE = 0.05  # 5% penalty on overdue amounts
    REMINDER_DAYS = 5  # Days before due date to send reminder
    
    def __init__(self, policyholder, product, amount, due_date=None):
        self.transaction_id = self._generate_transaction_id()
        self.policyholder = policyholder
        self.product = product
        self.amount = amount
        self.due_date = due_date or (datetime.now() + timedelta(days=30))
        self.payment_date = None
        self.status = PaymentStatus.PENDING
        self.created_date = datetime.now()
        self.penalty_amount = 0.0
    
    @classmethod
    def _generate_transaction_id(cls):
        cls._transaction_id_counter += 1
        return f"TXN{cls._transaction_id_counter}"
    
    def process_payment(self):
        if self.status == PaymentStatus.COMPLETED:
            print(f"Payment {self.transaction_id} has already been processed.")
            return False
        
        if self.amount <= 0:
            print("Payment amount must be greater than zero.")
            self.status = PaymentStatus.FAILED
            return False
        
        self.payment_date = datetime.now()
        self.status = PaymentStatus.COMPLETED
        
        payment_record = {
            "transaction_id": self.transaction_id,
            "product_name": self.product.product_name,
            "amount": self.amount,
            "payment_date": self.payment_date.strftime("%Y-%m-%d %H:%M:%S"),
            "penalty": self.penalty_amount
        }
        self.policyholder.add_payment_record(payment_record)
        
        print(f"Payment {self.transaction_id} of ${self.amount:.2f} processed successfully for {self.policyholder.name}.")
        return True
    
    def check_overdue(self):
        if self.status == PaymentStatus.COMPLETED:
            return False
        
        if datetime.now() > self.due_date:
            self.status = PaymentStatus.OVERDUE
            days_overdue = (datetime.now() - self.due_date).days
            self.penalty_amount = self.amount * self.PENALTY_RATE * (days_overdue / 30)
            print(f"Payment {self.transaction_id} is OVERDUE by {days_overdue} days. Penalty: ${self.penalty_amount:.2f}")
            return True
        
        return False
    
    def send_payment_reminder(self):
        if self.status in [PaymentStatus.COMPLETED, PaymentStatus.OVERDUE]:
            return False
        
        days_until_due = (self.due_date - datetime.now()).days
        
        if 0 <= days_until_due <= self.REMINDER_DAYS:
            print(f"[REMINDER] Dear {self.policyholder.name}, your payment of ${self.amount:.2f} "
                  f"for {self.product.product_name} is due in {days_until_due} days.")
            return True
        
        return False
    
    def apply_penalty(self):
        self.check_overdue()
        total_due = self.amount + self.penalty_amount
        print(f"Penalty applied to {self.transaction_id}. Total due: ${total_due:.2f}")
        return total_due
    
    def get_payment_details(self):
        return {
            "transaction_id": self.transaction_id,
            "policyholder": self.policyholder.name,
            "product": self.product.product_name,
            "amount": self.amount,
            "penalty": self.penalty_amount,
            "total_due": self.amount + self.penalty_amount,
            "due_date": self.due_date.strftime("%Y-%m-%d"),
            "payment_date": self.payment_date.strftime("%Y-%m-%d") if self.payment_date else "Not paid",
            "status": self.status.value,
            "created_date": self.created_date.strftime("%Y-%m-%d %H:%M:%S")
        }
    
    def __repr__(self):
        return f"Payment(ID={self.transaction_id}, Amount=${self.amount:.2f}, Status={self.status.value})"